# 03 - Prototype AM Training (smoke test)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enph-479-edge-ai-stt/enph-479-edge-ai-stt/blob/main/training/notebooks/03_prototype_am_training.ipynb)

First end-to-end run of the acoustic model on a **small subset**. The goal is to prove the pipeline works, **not** to get a good model:

1. overfit one tiny batch to near-zero CTC loss (wiring is correct),
2. CTC loss drops past the blank wall on the subset,
3. checkpoint + resume survive a killed session,
4. greedy dev CER is computable.

Real code lives in `training/src/training/{vocab,features,am}/`; this notebook is a thin launcher. A T4 helps but CPU works for the smoke. Provisional specs (30-symbol vocab, no peephole) are still pending the `shared/specs` freeze.

## 1. Pull the repo into the VM

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/enph-479-edge-ai-stt/enph-479-edge-ai-stt.git"
REPO_DIR = "/content/enph-479-edge-ai-stt"
TRAINING_DIR = os.path.join(REPO_DIR, "training")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

# Editable install pulls jiwer; torch and torchaudio are already on Colab.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", TRAINING_DIR], check=True)

# The editable install registers via a .pth file Python only reads at startup, so put
# src/ on the path for this already-running kernel.
sys.path.insert(0, os.path.join(TRAINING_DIR, "src"))
importlib.invalidate_caches()

training = importlib.import_module("training")
print("package:", training.__file__)

## 2. Get a small subset

`dev-clean` (~337 MB) is small and enough for a smoke test. We carve a train slice and a disjoint dev slice out of it; the real run uses `train-clean-100` for training and keeps `dev-clean` for tuning.

In [ ]:
from training.am.dataset import list_utterances
from training.data.librispeech import download_subset

DATA_DIR = "/content/data"
subset_dir = download_subset("dev-clean", dest=DATA_DIR)

items = list_utterances(subset_dir)
print(f"{len(items)} utterances in dev-clean")

N_TRAIN, N_DEV = 200, 50  # smoke sizes; raise once the pipeline is proven
train_items = items[:N_TRAIN]
dev_items = items[N_TRAIN : N_TRAIN + N_DEV]
print(f"train {len(train_items)}, dev {len(dev_items)}")

## 3. Overfit one batch (wiring sanity)

Two utterances, driven to near-zero CTC loss. If this does not fall, the loss/gradient/CTC wiring is wrong and there is no point training on the subset. A tiny model overfits fastest and still exercises the full path.

In [ ]:
from training.am.train import TrainConfig, overfit_one_batch

smoke_cfg = TrainConfig(n_hidden=128, n_layers=2, epochs=8, batch_size=8)
losses = overfit_one_batch(smoke_cfg, train_items, n_utts=2, steps=150)
print(f"overfit loss {losses[0]:.2f} -> {losses[-1]:.2f}")
assert losses[-1] < 0.5 * losses[0], "CTC wiring problem: loss did not drop"

## 4. Compute CMVN, then train

CMVN mean/std come from the **train slice only** and are reused for dev. `train()` checkpoints atomically and resumes from the latest checkpoint, so re-running this cell picks up where a killed session left off (that is the resume test). Expect the blank wall: near-100% CER for the first epochs, then it falls.

In [ ]:
from training.am.dataset import compute_cmvn_over
from training.am.train import train

mean, std = compute_cmvn_over(train_items)
model, best_cer = train(smoke_cfg, train_items, dev_items, mean, std, resume=True)
print(f"best dev CER {best_cer:.3f}")

## 5. Sanity checks

In [ ]:
from pathlib import Path

ckpt = Path(smoke_cfg.ckpt_dir) / "latest.pt"
assert ckpt.exists(), "no checkpoint written"
print("checkpoint:", ckpt, f"({ckpt.stat().st_size / 1e6:.1f} MB)")
print("Smoke passed: loss dropped, checkpoint + resume work, CER computed.")